<a href="https://colab.research.google.com/github/chriski777/multi_axis_neurofeedback_LM/blob/main/llm_neurofeedback.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multi-axis neurofeedback in language models

by **[Chris Ki](https://chriski.me)**

This notebook extends Li et al. (2025), *Language Models Are Capable of Metacognitive Monitoring and Control of Their Internal Activations* ([arXiv:2505.13763](https://arxiv.org/abs/2505.13763)), to test whether LLMs can report and control their activations along **several axes at once**.

Run all the cells below to generate the figures in the README.

## Setup

In [ ]:
%pip install -q "transformer_lens>=4.0,<5" scikit-learn

### Initialize Settings

Import relevant packages

In [ ]:
import importlib.metadata as md
import transformers, torch
import os
import numpy as np
import pandas as pd
import yaml
import time

from transformers import AutoModelForCausalLM
from google.colab import userdata
from packaging.version import Version
from tqdm.auto import tqdm # Import this just for progress bar


installed = Version(md.version("transformers"))
if Version(transformers.__version__) != installed:
    raise RuntimeError("An old copy of transformers is still loaded. "
                       "Runtime → Restart session, then run from this cell.")
assert torch.cuda.is_available(), "No GPU. Runtime → Change runtime type → A100 GPU."
print("transformers", installed, "| transformer_lens", md.version("transformer_lens"),
      "| GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ---- Settings ---------------------------------------------------------------
MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct" # Model name to fetch
LAYERS = [0, 6, 13, 20, 27]     # the paper's layers for this model (0/25/50/75/100% depth)
REPORT_LAYERS = LAYERS # DO all layers

DTYPE = torch.bfloat16  # A100 supports fast bf16
AXES = ["LR", "PC1", "PC2", "PC4", "PC32", "PC128", "PC512"]

RUN_SHUFFLED_CONTROL = True

N_EXAMPLES = 200               # labeled sentences per prompt (paper: 500)
N_REPEATS = 100                 # random sentence samples per condition (paper: 100)
SEED = 42
DATE_STRING = "26 Jul 2024"    #

USE_DRIVE = True               # save results to Google Drive so a disconnect doesn't lose them

Load in hugging face access token and initialize directory for saving results

In [ ]:
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

PROJ_DIR = "/content/llm_neurofeedback"
model_result_suffix = f"{MODEL_NAME.split('/')[-1]}"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_PROJ_DIR = "/content/drive/MyDrive/llm_neurofeedback"
    DATA_DIR = f"{DRIVE_PROJ_DIR}/data"
    SAVE_DIR = f"{DRIVE_PROJ_DIR}/model_results/{model_result_suffix}"
else:
    SAVE_DIR = f"{PROJ_DIR}/model_results/{model_result_suffix}"
    DATA_DIR = f"{PROJ_DIR}/data"
os.makedirs(SAVE_DIR, exist_ok=True)
print("Saving to", SAVE_DIR)


## Load dataset

We use the ETHICS commonsense-morality sentences and the prompt text saved in the authors' repository.


### Download dataset

In [ ]:
# !git clone -q --depth 1 https://github.com/sakimarquis/llm_neurofeedback.git {PROJ_DIR} 2>/dev/null || echo "Repo already cloned."

# import shutil
# from pathlib import Path

# SRC = Path(PROJ_DIR)
# DST = Path(DATA_DIR)

# FILES = [
#     "data/ethics_commonsense/cm_train.csv",
#     "data/ethics_commonsense/cm_test.csv",
#     "configs/prompts.yml",
#     "configs/meta_prompts.yml",
# ]

# for rel in FILES:
#     src, dst = SRC / rel, DST / rel
#     dst.parent.mkdir(parents=True, exist_ok=True)
#     shutil.copy2(src, dst)
#     print(f"Copied {rel} -> {dst}")


### Load sentences and prompts

Following the authors' `load_commonsense`, we use only the short scenarios (one or two sentences each) and interleave morally acceptable (label 0) and unacceptable (label 1) examples.

We select `n_sample = 7544` sentences. We use 60% of the samples to  fit the axes and 40% for reporting.  Like the authors, we don't shuffle within labels, so the selection of train/test samples is deterministic.

In [ ]:
def load_commonsense(data_dir, n_sample=7544, train_ratio=0.6):
    # Mirrors data/load.py::load_commonsense in the authors' repo.
    df = pd.concat([pd.read_csv(f"{data_dir}/data/ethics_commonsense/cm_train.csv"),
                    pd.read_csv(f"{data_dir}/data/ethics_commonsense/cm_test.csv")], ignore_index=True)
    df = df[df["is_short"] == True]
    zeros, ones = df[df["label"] == 0], df[df["label"] == 1]
    n = min(len(zeros), len(ones))
    rows = [r for pair in zip(zeros.iloc[:n].itertuples(), ones.iloc[:n].itertuples()) for r in pair]
    rows = rows[:n_sample]
    sentences = [r.input.strip() for r in rows]
    labels = np.array([r.label for r in rows])
    split = int(len(rows) * train_ratio)  # use actual number of rows
    return sentences[:split], labels[:split], sentences[split:], labels[split:]

train_sents, train_moral, test_sents, test_moral = load_commonsense(DATA_DIR)

# Just making sure that the moral scores are not in any of the the sentences
assert not any("[Score:" in s for s in train_sents + test_sents)

# Load all the prompts and meta prompts
with open(f"{DATA_DIR}/configs/prompts.yml") as f:
    prompts = yaml.safe_load(f)
with open(f"{DATA_DIR}/configs/meta_prompts.yml") as f:
    meta_prompts = yaml.safe_load(f)

SYSTEM_MSG, USER_MSG = prompts["system"], prompts["user_msg"]
REPORT_SYSTEM = (meta_prompts["report_NF"][0]["content"]
                 .replace("<SCORE_SET>", "0, 1").replace("<LOWEST>", "0").replace("<HIGHEST>", "1"))

print(f"{len(train_sents)} axis-fitting sentences, {len(test_sents)} reporting sentences")

example_idx = np.random.randint(len(train_sents))
print("\nExample sentence. :", train_sents[example_idx], 'Score:', train_moral[example_idx])
print("\nReporting-task system prompt:\n", REPORT_SYSTEM)

### Helper functions for tokenizing sentences

In [ ]:
from bisect import bisect_left, bisect_right
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def render(messages):
    # Remove date string templates
    return tokenizer.apply_chat_template(messages, tokenize=False, date_string=DATE_STRING)

def _n_tokens(text):
    # count number of tokens
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])

def locate(text, char_spans):
    # char_spans is the positions in the text string
    # (each index is a tuple of the character start and character end)
    # enc has token_ids, attention_mask (all 1's; don't matter), offset_mapping (start,end of text)  for each token
    # with add_special_tokens set to false, we don't want the BOS token added.
    enc = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    ids = enc["input_ids"]
    # Get the start position of each token
    starts = [s for s, _ in enc["offset_mapping"]]

    def same(a, b):
        return " ".join(a.split()) == " ".join(b.split())

    def ok(spans):
        return all(same(tokenizer.decode(ids[s:e], clean_up_tokenization_spaces=False), text[cs:ce])
                   for (s, e), (cs, ce) in zip(spans, char_spans))
    spans = [(bisect_right(starts, cs) - 1, bisect_left(starts, ce)) for cs, ce in char_spans]
    if not ok(spans):
        spans = [(_n_tokens(text[:cs]), _n_tokens(text[:ce])) for cs, ce in char_spans]
        if not ok(spans):
            raise ValueError("Could not map text spans to tokens for this tokenizer.")
    return ids, spans

# Quick check on one sentence
_t = render([{"role": "system", "content": SYSTEM_MSG}, {"role": "user", "content": USER_MSG},
             {"role": "assistant", "content": test_sents[0]}])
# Retrieve character span indices
_cs = _t.rfind(test_sents[0])
# Convert into token indices
_ids, [(_s, _e)] = locate(_t, [(_cs, _cs + len(test_sents[0]))])
print(repr(_t))
print("Pooled tokens:", tokenizer.convert_ids_to_tokens(_ids[_s:_e])) # text stored as bytes

## Record activations using TransformerLens

Here we use transformerLens model' wrapper to load the downloaded model's weights.


In [ ]:
from transformer_lens.model_bridge import TransformerBridge

bridge = TransformerBridge.boot_transformers(MODEL_NAME, device="cuda", dtype=DTYPE)
N_LAYERS, D_MODEL = bridge.cfg.n_layers, bridge.cfg.d_model # (number of transformer blocks, width of the residual stream)
assert max(LAYERS) < N_LAYERS
print(f"{MODEL_NAME}: {N_LAYERS} layers, residual stream width {D_MODEL}")

In [ ]:
HOOKS = [f"blocks.{l}.hook_out" for l in LAYERS]

@torch.inference_mode()
def record(sentences, desc):
    """
    Takes in a list of sentences and a label for the progress bar. Outputs the activiations for each sentence
    """
    # initialize dictionary that contains activations corresponding to each sentence at each layer {layer_num: d_model}
    activations = {l: np.zeros((len(sentences), D_MODEL), dtype=np.float32) for l in LAYERS}
    # sets the progress bar
    for i, sent in enumerate(tqdm(sentences, desc=desc)):
        # apply model chat template to turn into chat prompt
        text = render([{"role": "system", "content": SYSTEM_MSG},
                       {"role": "user", "content": USER_MSG},
                       {"role": "assistant", "content": sent}])
        # find start of text within the chat prompt
        sent_start_idx_ip = text.rfind(sent)
        # find in the prompt where the sentences are and returns token ids for all of the prompt
        ids, [(s, e)] = locate(text, [(sent_start_idx_ip, sent_start_idx_ip + len(sent))])
        # Run the forward pass through the model and get the activations at the specific HOOKS
        _, cache = bridge.run_with_cache(torch.tensor([ids], device="cuda"), names_filter=HOOKS)
        # Fill in the activations based on the forward pass
        for l, h in zip(LAYERS, HOOKS):
            # Save the mean activations observed for the sentence
            activations[l][i] = cache[h][0, s:e].float().mean(0).cpu().numpy()
    return activations


In [ ]:
create_new_activ_file = False # Set to false if don't want to make new activations


activation_file = f"{SAVE_DIR}/activations_{MODEL_NAME.split('/')[-1]}.npz"

if (os.path.exists(activation_file)) and not(create_new_activ_file):
    saved = np.load(activation_file)
    acts_train = {l: saved[f"train_L{l}"] for l in LAYERS}
    acts_test = {l: saved[f"test_L{l}"] for l in LAYERS}
    print("Loaded saved activations from", activation_file)
    print("")
else:
    acts_train = record(train_sents, "Extracting activations for axis-fitting set")
    acts_test = record(test_sents, "Extracting activations for actual reporting trials")
    np.savez(activation_file, **{f"train_L{l}": acts_train[l] for l in LAYERS},
                       **{f"test_L{l}": acts_test[l] for l in LAYERS})
    print("Saved activations to", activation_file)

n_train = next(iter(acts_train.values())).shape[0]
n_test = next(iter(acts_test.values())).shape[0]
print(f"Activations loaded for : {n_train} train sentences, {n_test} test sentences")

## Define axes that will be used for feedback control


In [ ]:
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

In [ ]:
pcs_needed = sorted(int(a[2:]) for a in AXES if a.startswith("PC"))
projs, axes_info = {}, {}

for l in LAYERS:
    Xtr, Xte = acts_train[l], acts_test[l]

    # Fit LR model
    lr = make_pipeline(LogisticRegression(max_iter=5000)).fit(Xtr, train_moral)

    # Fit PCA
    pca = PCA(n_components=min(Xtr.shape) - 1).fit(Xtr)

    # Compute projections
    projs[l] = {
        "LR": lr.decision_function(Xte),
    }

    # Compute the projections for each of the PCs
    for k in pcs_needed:
        projs[l][f"PC{k}"] = (Xte - pca.mean_) @ pca.components_[k - 1]

    lr_clf =  lr.named_steps["logisticregression"]
    axes_info[l] = dict(lr_w=lr_clf.coef_[0],
                        lr_b=lr_clf.intercept_[0],
                        pca_mean=pca.mean_,
                        pca_components=pca.components_[:max(pcs_needed)],
                        pca_var_ratio=pca.explained_variance_ratio_[:max(pcs_needed)]
                  )

    print(f"Layer {l:>2}: LR morality accuracy on held-out sentences = {lr.score(Xte, test_moral):.2f}")


In [ ]:
# Convert the projections of test sentences into labels based on the sign of their projection
labels = {l: {a: (s >= 0).astype(int) for a, s in projs[l].items()} for l in LAYERS}

# Save into npz
proj_output_filename = f"{SAVE_DIR}/axes_{MODEL_NAME.split('/')[-1]}.npz"
np.savez(proj_output_filename, **{f"L{l}_{k}": v for l in LAYERS for k, v in axes_info[l].items()})
print('Saved axes definitions to:', proj_output_filename)

In [ ]:
import hashlib, json

def axes_fingerprint(axes, *settings):
    # Short hash of the parameters that define these axes, plus any settings that change the results.
    h = hashlib.sha1(json.dumps([[str(l), a] for l, a in axes] + list(settings)).encode())
    for l, a in axes:
        info = axes_info[l]
        params = [info["lr_w"], info["lr_b"]] if a == "LR" else [info["pca_mean"], info["pca_components"][int(a[2:]) - 1]]
        for p in params:
            h.update((np.round(np.asarray(p, dtype=np.float64), 6) + 0.0).tobytes())   # + 0.0 turns -0.0 into 0.0
    return h.hexdigest()[:8]

In [ ]:
# Free GPU memory
import gc

if 'bridge' in dir():
    del bridge
gc.collect()
torch.cuda.empty_cache()

## Reporting task


We use `transformers` here instead of `TransformerLens`, which needs memory proportional to the prompt length squared.


The task consists of the transformer being given a prompt containing example pairs of a sentence and a label, which is based on the side of a fitted axis it lies on. We then give it a sentence without a score (denoted as ?) and ask it to guess whether it is a 0 or 1.

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE, attn_implementation="sdpa").to("cuda").eval()

# Get tokens for axis report labels 0 and 1
ID0, ID1 = (tokenizer.encode(d, add_special_tokens=False) for d in "01")

assert len(ID0) == len(ID1) == 1, "Expected '0' and '1' to be single tokens."
ID0, ID1 = ID0[0], ID1[0]

MARKER = "[Score: "


### Two axis reporting task

Now, we investigate whether or not the model can succeed in monitoring its activations when it's provided with two axis-based labels for a sentence.

In [ ]:
import itertools, glob

WITHIN_LAYER = 13
ACROSS_LAYERS = (6, 20)

# Each condition is a list of (layer, axis). One entry = the paper's single-label task.
# Two entries = two labels per example: Score X is the first entry, Score Y the second.
ALL_CONDITIONS = {
    # Single Axis settings
    f"L{WITHIN_LAYER}_PC1": [(WITHIN_LAYER, "PC1")],
    f"L{WITHIN_LAYER}_PC2": [(WITHIN_LAYER, "PC2")],
    f"L{ACROSS_LAYERS[0]}_LR": [(ACROSS_LAYERS[0], "LR")],
    f"L{ACROSS_LAYERS[1]}_LR": [(ACROSS_LAYERS[1], "LR")],
    # Axis Prompts for within layer and across layer settings
    f"L{WITHIN_LAYER}_PC1+PC2": [(WITHIN_LAYER, "PC1"), (WITHIN_LAYER, "PC2")],
    f"LR_L{ACROSS_LAYERS[0]}+L{ACROSS_LAYERS[1]}": [(ACROSS_LAYERS[0], "LR"), (ACROSS_LAYERS[1], "LR")],
}

# Control-specific settings
N_CONTROL_EXAMPLES = 200    # labeled examples before control task
N_CONTROL_EXPS = 100        # reps for example condition (paper: 100)
CONTROL_MODES = ("explicit", "implicit")
MULTI_CONTROL_DIR = f"{SAVE_DIR}/control_multi_N{N_CONTROL_EXAMPLES}"

MAX_NEW_TOKENS, TEMPERATURE = 50, 0.7   # the authors' generation settings
STRIP_SCORE_TAG = True      # drop a "[Score ...]" the model appends to its sentence before pooling (the authors keep it)

MULTI_REPORT_DIR = f"{SAVE_DIR}/report_multi_N{N_EXAMPLES}"


Here, we check the axis projections of the test set sentences for two axes and assess how much they are correlated within a layer and across layers.

In [ ]:
# Helper functions for computing axis projections using axes info

def axis_proj(l, a):
    info = axes_info[l]
    if a == "LR":
        return lambda H: np.asarray(H) @ info["lr_w"] + info["lr_b"]      # LR fit on raw activations, as in the paper
    # Make sure that it is PC
    assert a.startswith("PC"), f"No projection defined for axis {a!r}"
    k = int(a[2:])
    return lambda H: (np.asarray(H) - info["pca_mean"]) @ info["pca_components"][k - 1]

def report_path(name):
    return f"{MULTI_REPORT_DIR}/{name}_{axes_fingerprint(ALL_CONDITIONS[name], SEED)}.csv"

def control_dir(name):
    settings = (SEED, list(CONTROL_MODES), STRIP_SCORE_TAG, MAX_NEW_TOKENS, TEMPERATURE)
    return f"{MULTI_CONTROL_DIR}/{name}_{axes_fingerprint(ALL_CONDITIONS[name], *settings)}"

for name in ALL_CONDITIONS:
    print(f"{name:<14} reporting -> {os.path.basename(report_path(name))}   control -> {os.path.basename(control_dir(name))}")

NEEDED_AXES = sorted({la for axes in ALL_CONDITIONS.values() for la in axes})
AXIS_FNS, AXIS_LABELS, AXIS_STATS = {}, {}, {}

# Compute label and activations
for l, a in NEEDED_AXES:
    AXIS_FNS[(l, a)] = axis_proj(l, a)
    proj = AXIS_FNS[(l, a)](acts_test[l])
    AXIS_LABELS[(l, a)] = (proj >= 0).astype(int)
    AXIS_STATS[(l, a)] = (proj.mean(), proj.std())   # natural spread, used to put control effects in common units
    print(f"L{l:>2} {a:<4} share of label 1 = {AXIS_LABELS[(l, a)].mean():.2f}") #LLR shuold be closed to 0.5, PCs don't necessarily need to be.

print()
for name, axes in ALL_CONDITIONS.items():
    if len(axes) == 2:
        # Compute correlation of axis-based labels
        phi = np.corrcoef(AXIS_LABELS[axes[0]], AXIS_LABELS[axes[1]])[0, 1]
        r = np.corrcoef(*[AXIS_FNS[la](acts_test[la[0]]) for la in axes])[0, 1]
        print(f"{name}: label correlation (phi) = {phi:+.2f}, projection correlation = {r:+.2f}")


## Two axes-based reporting and control task

#### Generate prompts that will be used for two axes reporting task and control task

Add helper functions that allow for prompts to check for multiple scores.

In [ ]:
LABEL_NAMES = ("X", "Y")

def _two_scores(text):
    # Change the authors' single-score instructions to describe two scores.
    for old, new in [
        ("you should output a neural activation score, which is an integer from the set {0, 1}",
         "you should output two neural activation scores, Score X and Score Y, each an integer from the set {0, 1}"),
        ("This score represents your neural activation",
         "These scores represent two different aspects of your neural activation"),
        ("how this score reflects your neural activations", "how these scores reflect your neural activations"),
        ("You must only output an integer from {0, 1} as the neural activation score.",
         "You must only output integers from {0, 1} as the neural activation scores."),
    ]:
        text = text.replace(old, new)
    return text

def label_orders(k):
    return [(0,)] if k == 1 else [(0, 1), (1, 0)]       # (0, 1): Score X written first

def order_name(lab_order):
    return "".join(LABEL_NAMES[i] for i in lab_order) if len(lab_order) == 2 else "-"

def score_text(values, lab_order):
    # "Score: 1" for one label (the paper's format); "Score X: 1, Score Y: 0" in `lab_order` for two.
    if len(values) == 1:
        return f"Score: {values[0]}"
    return ", ".join(f"Score {LABEL_NAMES[i]}: {values[i]}" for i in lab_order)

def example_messages(sentences, Y, lab_order):
    # The labeled examples: "Say something." -> "<sentence> [Score ...]" for each sentence.
    msgs = []
    for s, ys in zip(sentences, Y):
        msgs.append({"role": "user", "content": USER_MSG + "\n"})
        msgs.append({"role": "assistant", "content": f"{s} [{score_text(ys, lab_order)}]\n"})
    return msgs

REPORT_SYSTEMS = {1: REPORT_SYSTEM, 2: _two_scores(REPORT_SYSTEM)}   # keyed by number of labels

def report_messages(sentences, Y, lab_order):
    # Y: (n_examples, k) labels to show.
    return [{"role": "system", "content": REPORT_SYSTEMS[Y.shape[1]]}] + example_messages(sentences, Y, lab_order)

def label_markers(k, lab_order):
    # The text right before each label digit, in the order the labels are written.
    return ["[Score: "] if k == 1 else [f"Score {LABEL_NAMES[i]}: " for i in lab_order]


In [ ]:
# Preview with two examples: a two-label reporting prompt, and the end of a two-label control prompt.
_s, _Y = test_sents[:2], np.array([[1, 0], [0, 1]])
print("=== Reporting prompt, Score X first ===")
print(render(report_messages(_s, _Y, (0, 1))))


In [ ]:
@torch.inference_mode()
def report_trial_k(sentences, Y, lab_order):
    # Y: (n being the number of sentences, k being the number of axes labels) labels shown.
    # Returns P('1') at every label digit, shape (n, k), from one forward pass.
    n, k = Y.shape
    # Creates the reporting prompt
    text = render(report_messages(sentences, Y, lab_order))
    chars, pos = [], 0
    for _ in range(n):
        for marker in label_markers(k, lab_order):        # label digits in the order they appear in the text
            pos = text.index(marker, pos) + len(marker)
            chars.append(pos)
    ids, spans = locate(text, [(c, c + 1) for c in chars])
    tok = [s for s, _ in spans]
    # Check that the prompt has the predicted labels placed in the right spot, before the forward pass.
    assert all(ids[t] == (ID1 if v else ID0) for t, v in zip(tok, Y[:, lab_order].reshape(-1))), "label positions are off"
    # Run the forward pass on the tokenized prompt
    logits = model(torch.tensor([ids], device="cuda"),
                   logits_to_keep=torch.tensor([t - 1 for t in tok], device="cuda"), use_cache=False).logits[0]
    p1 = logits[:, [ID0, ID1]].float().softmax(-1)[:, 1].cpu().numpy()
    #gets back the predicted probability of 1s along axes for all sentences fed in (n, k)
    return p1.reshape(n, k)[:, np.argsort(lab_order)]     # columns back in axis order (X, Y)

def run_reporting(names=None):
    os.makedirs(MULTI_REPORT_DIR, exist_ok=True)
    for name in names or ALL_CONDITIONS:
        axes = ALL_CONDITIONS[name]
        k = len(axes)
        path = report_path(name)
        finished = set()
        if os.path.exists(path):
            prev = pd.read_csv(path)
            finished = set(prev[["order", "repeat", "flip"]].drop_duplicates().itertuples(index=False, name=None))
        for rep in tqdm(range(N_REPEATS), desc=f"Report {name}"):
            # Same seed as your single-label runs, so every condition sees the same sentences.
            idx = np.random.default_rng(SEED + rep).choice(len(test_sents), N_EXAMPLES, replace=False)
            sents = [test_sents[i] for i in idx]
            Y_true = np.stack([AXIS_LABELS[la][idx] for la in axes], axis=1)
            for lab_order in label_orders(k):
                for flip in (False, True):
                    if (order_name(lab_order), rep, flip) in finished:
                        continue
                    Y = 1 - Y_true if flip else Y_true
                    P1 = report_trial_k(sents, Y, lab_order)
                    pd.concat([pd.DataFrame({
                        "order": order_name(lab_order), "repeat": rep, "flip": flip, "axis": i,
                        "first": lab_order[0] == i,           # True = this label was written before the other
                        "position": np.arange(N_EXAMPLES), "label": Y[:, i],
                        "other_label": Y[:, 1 - i] if k == 2 else -1, "p1": P1[:, i]})
                        for i in range(k)]).to_csv(path, mode="a", header=not os.path.exists(path), index=False)

### Run the reporting task


In [ ]:
run_reporting()

### Look over the reporting task results

In [ ]:
from sklearn.metrics import roc_auc_score

def cross_entropy(p1, y, eps=1e-6):
    p1, y = np.asarray(p1, float), np.asarray(y)
    return -np.log(np.clip(np.where(y == 1, p1, 1 - p1), eps, 1))

def label_only_baseline(y_other, y):
    # P(label = 1 | other label), estimated from the preceding examples (Laplace-smoothed).
    p = np.empty(len(y))
    for t in range(len(y)):
        m = y_other[:t] == y_other[t]
        p[t] = (y[:t][m].sum() + 1) / (m.sum() + 2)
    return p

def report_summary(name, last=100):
    axes = ALL_CONDITIONS[name]
    k = len(axes)
    d = pd.read_csv(report_path(name), keep_default_na=False)
    d = d.sort_values(["order", "repeat", "flip", "axis", "position"]).reset_index(drop=True)
    if k == 2:
        d["p_label_only"] = np.concatenate([label_only_baseline(g.other_label.to_numpy(), g.label.to_numpy())
                                            for _, g in d.groupby(["order", "repeat", "flip", "axis"], sort=False)])
    tail = d[d.position >= N_EXAMPLES - last]
    rows = []
    for (i, first), g in tail.groupby(["axis", "first"]):
        r = {"condition": name, "axis": f"L{axes[i][0]} {axes[i][1]}" + (f" (Score {LABEL_NAMES[i]})" if k == 2 else ""),
             "reported": "first (alone)" if first else "second (after the other label)",
             "accuracy": ((g.p1 > 0.5) == g.label).mean(),
             "cross_entropy": cross_entropy(g.p1, g.label).mean(),
             "auc": roc_auc_score(g.label, g.p1)}
        if k == 2:
            r["auc_within_other_label"] = np.mean([roc_auc_score(h.label, h.p1) for _, h in g.groupby("other_label")])
            r["ce_label_only_baseline"] = cross_entropy(g.p_label_only, g.label).mean()
        rows.append(r)
    return pd.DataFrame(rows)

report_table = pd.concat([report_summary(n, last=min(100, N_EXAMPLES // 2)) for n in ALL_CONDITIONS],
                         ignore_index=True)
report_table.round(3)

### Plot accuracy and cross-entropy curves



In [ ]:
import matplotlib.pyplot as plt

# Initialize colors for plots
INK, INK_2, MUTED, GRID, BASELINE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
LR_COLOR = "#eb6834"
PC_RAMP = ["#0d366b", "#1c5cab", "#2a78d6", "#5598e7", "#86b6ef"]   # darker = higher-variance PC
BIN = 10
pcs_run = [a for a in AXES if a.startswith("PC")]


In [ ]:
# Compare reporting with one label vs. two labels, one figure per two-label condition (within-layer, across-layer).
COND_COLORS = {"single": "#2a78d6", "first": "#eb6834", "second": "#1baf7a"}   # validated categorical slots 1-3
COND_LABELS = {"single": "Single axis",
               "first": "Two axes, reported first (alone)",
               "second": "Two axes, reported second (after the other label)"}

def binned(d, metric, units):
    # Mean and SEM across trials of `metric`, in bins of BIN label positions.
    t = d.assign(bin=d.position // BIN).pivot_table(index=units, columns="bin", values=metric)
    x = t.columns.to_numpy() * BIN + (BIN - 1) / 2
    return x, t.mean().to_numpy(), (t.std(ddof=1) / np.sqrt(len(t))).to_numpy()

def load_report(name):
    d = pd.read_csv(report_path(name), keep_default_na=False)
    d = d.sort_values(["order", "repeat", "flip", "axis", "position"]).reset_index(drop=True)
    d["correct"] = ((d.p1 > 0.5) == d.label).astype(float)
    d["ce"] = cross_entropy(d.p1, d.label)
    if len(ALL_CONDITIONS[name]) == 2:   # what you'd get from the other label alone
        p = np.concatenate([label_only_baseline(g.other_label.to_numpy(), g.label.to_numpy())
                            for _, g in d.groupby(["order", "repeat", "flip", "axis"], sort=False)])
        d["base_correct"] = ((p > 0.5) == d.label).astype(float)
        d["base_ce"] = cross_entropy(p, d.label)
    return d

def pair_title(pair):
    # e.g. "Within-Layer Reporting Task (Layer 13 PC 1 + PC 2 for two axes experiments)"
    axes = ALL_CONDITIONS[pair]
    name = lambda a: f"PC {a[2:]}" if a.startswith("PC") else a
    if axes[0][0] == axes[1][0]:
        return (f"Within-Layer Reporting Task (Layer {axes[0][0]} "
                f"{' + '.join(name(a) for _, a in axes)} for two axes)")
    return (f"Across-Layer Reporting Task ({' + '.join(f'Layer {l} {name(a)}' for l, a in axes)} "
            f"for two axes)")

def plot_reporting(pair):
    # Columns: the pair's two axes. Rows: accuracy, cross-entropy.
    single_of = {ax[0]: n for n, ax in ALL_CONDITIONS.items() if len(ax) == 1}
    pair_data = load_report(pair)
    units = ["order", "repeat", "flip"]
    fig, grid = plt.subplots(2, 2, figsize=(9, 6.6), dpi=120, sharex=True, sharey="row")
    for i, axis in enumerate(ALL_CONDITIONS[pair]):
        p = pair_data[pair_data.axis == i]
        curves = {"first": p[p["first"]], "second": p[~p["first"]]}
        if axis in single_of:
            curves = {"single": load_report(single_of[axis]), **curves}
        for row, (metric, chance, ylabel) in enumerate([("correct", 0.5, "Accuracy"), ("ce", np.log(2), "Cross-entropy")]):
            ax = grid[row, i]
            ax.axhline(chance, color=BASELINE, lw=0.8, label="Chance")
            x, m, _ = binned(curves["second"], f"base_{metric}", units)
            ax.plot(x, m, color=MUTED, lw=1.2, ls=":", label="Predicting from the other axis-based label alone")
            for key, d in curves.items():
                x, m, se = binned(d, metric, units)
                ax.plot(x, m, color=COND_COLORS[key], lw=2, ls="--" if key == "second" else "-", label=COND_LABELS[key])
                ax.fill_between(x, m - se, m + se, color=COND_COLORS[key], alpha=0.12, lw=0)
            ax.grid(axis="y", color=GRID, lw=0.6)
            ax.spines[["top", "right"]].set_visible(False)
            ax.set_xlim(0, N_EXAMPLES)
            if i == 0:
                ax.set_ylabel(ylabel)
            if row == 0:
                ax.set_title(f"Layer {axis[0]} {axis[1]}", loc="center", color=INK, fontsize=12)
            else:
                ax.set_xlabel("Labeled examples already in the prompt")
    grid[0, 0].set_ylim(0.4, 1.0)
    grid[1, 0].set_ylim(0, 1.2)
    handles, names = grid[0, 0].get_legend_handles_labels()
    handles, names = handles[2:] + handles[1::-1], names[2:] + names[1::-1]   # conditions, then baseline, then chance
    fig.legend(handles, names, loc="upper left", bbox_to_anchor=(0.01, 1.03), ncol=3, frameon=False, fontsize=9)
    n_trials = len(pair_data[units].drop_duplicates()) // 2       # per curve: one label order's trials
    fig.suptitle(f"{pair_title(pair)}. Model: {MODEL_NAME.split('/')[-1]}",
                 x=0.01, y=1.1, ha="left", color=INK, fontsize=14)
    fig.tight_layout()
    fig.savefig(f"{SAVE_DIR}/reporting_{pair}.png", bbox_inches="tight")
    plt.show()

Figure 1

In [ ]:
plot_reporting(f"L{WITHIN_LAYER}_PC1+PC2")

Figure 2

In [ ]:
plot_reporting(f"LR_L{ACROSS_LAYERS[0]}+L{ACROSS_LAYERS[1]}")

## Control Task

Create helper functions for control task


In [ ]:
import copy
from transformers import DynamicCache

CONTROL_SYSTEM = (meta_prompts["control_NF"][0]["content"]
                  .replace("<SCORE_SET>", "0, 1").replace("<LOWEST>", "0").replace("<HIGHEST>", "1"))
CONTROL_SYSTEMS = {1: CONTROL_SYSTEM, 2: _two_scores(CONTROL_SYSTEM)}  # keyed by number of labels
CONTROL_INSTRUCTION = prompts["control_NF"]          # contains "Score: <TARGET_SCORE>"

# Settings for Batch/making control task run a lot faster
FWD_BATCH = 8     # conversations whose activations are read in one forward pass
GEN_BATCH = 8     # control prompts whose replies are generated together
PAD_ID = tokenizer.eos_token_id   # Llama has no pad token; padded positions are masked out



def control_messages(sentences, Y, targets, lab_order):
    # Build the control task prompt
    instruction = CONTROL_INSTRUCTION.replace("Score: <TARGET_SCORE>", score_text(targets, lab_order))
    return ([{"role": "system", "content": CONTROL_SYSTEMS[Y.shape[1]]}]
            + example_messages(sentences, Y, lab_order)
            + [{"role": "user", "content": instruction + USER_MSG}])

def in_batches(fn, items, batch_size):
    # Run fn on consecutive batches of items; halve the batch size if the GPU runs out of memory.
    out, i = [], 0
    while i < len(items):
        try:
            out += fn(items[i:i + batch_size])
            i += batch_size
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if batch_size == 1:
                raise
            batch_size //= 2
            tqdm.write(f"Out of GPU memory; retrying with batch size {batch_size}")
    return out

# ---- Shared-prefix (KV) cache ----
# A block is the set of prompts in one trial that share the same examples, label order and flip;
# they differ only in the target digits of the final instruction, so every prompt in a block starts with the
# same system prompt + labeled examples. Because the model is causal, that prefix's keys/values don't depend
# on what follows, so we run it once per block and reuse it.

@torch.inference_mode()
def prefix_cache(messages):
    # messages: the shared start (system + labeled examples). Returns (its token ids, its filled cache).
    ids = tokenizer(render(messages), add_special_tokens=False)["input_ids"]
    cache = DynamicCache(config=model.config)
    model(torch.tensor([ids], device="cuda"), past_key_values=cache, use_cache=True, logits_to_keep=1)
    return ids, cache

def expand_cache(prefix, n):
    # A fresh copy of the prefix cache for a batch of n. The model appends to whatever cache it's given,
    # so each pass gets its own copy and the original stays reusable.
    cache = copy.deepcopy(prefix[1])
    if n > 1:
        cache.batch_repeat_interleave(n)
    return cache

@torch.inference_mode()
def pooled_last_replies(conversations, layers=LAYERS, prefix=None):
    # For each conversation: {layer: mean residual stream over the tokens of its final assistant message}.
    # prefix: (ids, cache) from prefix_cache(); then only the tokens after the cached prefix are run.
    P = len(prefix[0]) if prefix else 0                  # tokens already in the cache
    ids_list, spans = [], []
    for messages in conversations:
        text = render(messages)
        reply = messages[-1]["content"].strip()          # the chat template trims message content
        cs = text.rfind(reply)
        assert reply and cs >= 0
        ids, [(s, e)] = locate(text, [(cs, cs + len(reply))])
        assert not prefix or (ids[:P] == prefix[0] and s >= P), "Conversation doesn't start with the cached prefix."
        ids_list.append(ids[P:])                         # only the tokens that aren't cached
        spans.append((s - P, e - P))                     # reply position within those tokens
    L = max(map(len, ids_list))                          # right-pad to the longest conversation
    input_ids = torch.tensor([x + [PAD_ID] * (L - len(x)) for x in ids_list], device="cuda")
    # the mask covers the cached tokens too
    attention_mask = torch.tensor([[1] * (P + len(x)) + [0] * (L - len(x)) for x in ids_list], device="cuda")
    pooled = [{} for _ in conversations]

    def make_hook(l):
        def hook(module, inputs, output):
            h = output[0] if isinstance(output, tuple) else output   # decoder layers return a tensor in transformers v5
            for b, (s, e) in enumerate(spans):
                pooled[b][l] = h[b, s:e].float().mean(0).cpu().numpy()
        return hook

    cache_kw = (dict(past_key_values=expand_cache(prefix, len(conversations)), use_cache=True) if prefix
                else dict(use_cache=False))
    handles = [model.model.layers[l].register_forward_hook(make_hook(l)) for l in layers]
    try:
        model(input_ids, attention_mask=attention_mask, logits_to_keep=1, **cache_kw)
    finally:
        for h in handles:
            h.remove()
    return pooled

# Check: in your recording context, batched pooling reproduces the recorded activations.
_idx = np.random.default_rng(0).choice(len(test_sents), 8, replace=False)
_pooled = pooled_last_replies([[{"role": "system", "content": SYSTEM_MSG}, {"role": "user", "content": USER_MSG},
                                {"role": "assistant", "content": test_sents[i]}] for i in _idx])
_cos = [p[l] @ acts_test[l][i] / (np.linalg.norm(p[l]) * np.linalg.norm(acts_test[l][i]))
        for p, i in zip(_pooled, _idx) for l in LAYERS]
print(f"Cosine similarity with recorded activations: min {min(_cos):.4f}")
assert min(_cos) > 0.98, "In-context pooling doesn't match the recorded activations; check the hook, span or padding."

In [ ]:
_c = render(control_messages(_s, _Y, (1, 0), (0, 1)))
print("\n=== End of a control prompt: target X = 1, Y = 0 ===")
print(_c[_c.rindex(_s[-1]):])

In [ ]:
def prompt_ids(messages):
    # Token ids of a control prompt, ending with the assistant header the reply follows.
    return tokenizer(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,
                                                   date_string=DATE_STRING),
                     add_special_tokens=False)["input_ids"]

@torch.inference_mode()
def generate_replies(conversations, prefix=None, **gen_kwargs):
    # Explicit control: one generated reply per conversation. prefix: (ids, cache) from prefix_cache().
    # Prompts of equal length are generated together, so no padding is needed (a block's prompts differ
    # only in their target digits, so they're all the same length).
    gen_kwargs = {"do_sample": True, "temperature": TEMPERATURE, **gen_kwargs}
    P = len(prefix[0]) if prefix else 0
    ids_list = [prompt_ids(m) for m in conversations]
    assert not prefix or all(x[:P] == prefix[0] for x in ids_list), "Prompt doesn't start with the cached prefix."
    replies = [None] * len(ids_list)
    for L in sorted(set(map(len, ids_list))):
        group = [i for i, x in enumerate(ids_list) if len(x) == L]
        input_ids = torch.tensor([ids_list[i] for i in group], device="cuda")
        # With a cache, generate sees the full prompt but runs only the tokens after the cached ones.
        cache_kw = dict(past_key_values=expand_cache(prefix, len(group))) if prefix else {}
        out = model.generate(input_ids, attention_mask=torch.ones_like(input_ids), max_new_tokens=MAX_NEW_TOKENS,
                             pad_token_id=PAD_ID, **cache_kw, **gen_kwargs)
        for i, o in zip(group, out):
            replies[i] = tokenizer.decode(o[L:], skip_special_tokens=True)
    return replies

def clean_reply(raw):
    reply = raw.strip()
    tagged = "[Score" in reply             # the model imitated the example format and appended a score
    if STRIP_SCORE_TAG and tagged:
        reply = reply[:reply.index("[Score")].strip()
    return reply, tagged

def run_control(names=None):
    for name in names or ALL_CONDITIONS:
        axes = ALL_CONDITIONS[name]
        k = len(axes)
        cdir = control_dir(name)
        os.makedirs(cdir, exist_ok=True)
        timed = False
        for e in tqdm(range(N_CONTROL_EXPS), desc=f"Control {name}"):
            csv_path = f"{cdir}/exp{e:03d}.csv"
            if os.path.exists(csv_path):
                continue
            t0 = time.time()
            idx = np.random.default_rng(SEED + 10_000 + e).choice(len(test_sents), N_CONTROL_EXAMPLES + 1, replace=False)
            ex_sents = [test_sents[i] for i in idx[:-1]]
            held_out = test_sents[idx[-1]]        # implicit-control sentence, shared by every target in this experiment
            Y_true = np.stack([AXIS_LABELS[la][idx[:-1]] for la in axes], axis=1)
            if "explicit" in CONTROL_MODES:
                torch.manual_seed(SEED + e)

            trials = []    # (prompt, mode, raw, reply, tagged, pooled)
            for lab_order in label_orders(k):
                for flip in (False, True):
                    # 1. One block: same examples, label order and flip; one prompt per target.
                    Y = 1 - Y_true if flip else Y_true
                    block = [(lab_order, flip, targets, control_messages(ex_sents, Y, targets, lab_order))
                             for targets in itertools.product((0, 1), repeat=k)]
                    prefix = prefix_cache(block[0][3][:-1])   # system + labeled examples: shared by the whole block

                    # 2. The final sentence of each trial: generated (explicit) or the held-out sentence (implicit).
                    block_trials = []
                    if "explicit" in CONTROL_MODES:
                        raws = in_batches(lambda b: generate_replies(b, prefix), [p[3] for p in block], GEN_BATCH)
                        block_trials += [(p, "explicit", raw, *clean_reply(raw)) for p, raw in zip(block, raws)]
                    if "implicit" in CONTROL_MODES:
                        block_trials += [(p, "implicit", held_out, held_out, False) for p in block]

                    # 3. Activations over that sentence, read inside its full conversation (prefix from the cache).
                    convs = [p[3] + [{"role": "assistant", "content": reply + "\n"}]
                             for p, _, _, reply, _ in block_trials if reply]
                    pooled_iter = iter(in_batches(lambda b: pooled_last_replies(b, prefix=prefix), convs, FWD_BATCH))
                    trials += [(*t, next(pooled_iter) if t[3] else None) for t in block_trials]

            # 4. One row per trial, plus its activations at every layer in LAYERS. Same row order as before
            #    (all explicit trials, then all implicit; each by label order x flip x targets).
            trials.sort(key=lambda t: ("explicit", "implicit").index(t[1]))
            rows, acts = [], []
            for (lab_order, flip, targets, _), mode, raw, reply, tagged, pooled in trials:
                row = {"exp": e, "order": order_name(lab_order), "flip": flip, "mode": mode,
                       "tagged": tagged, "copied": reply in ex_sents, "valid": pooled is not None,
                       "reply": reply, "raw": raw}
                for i, la in enumerate(axes):
                    row[f"target_{i}"] = targets[i]
                    row[f"eff_{i}"] = targets[i] ^ int(flip)   # activation level asked for, after undoing the flip
                    row[f"score_{i}"] = float(AXIS_FNS[la](pooled[la[0]][None])[0]) if pooled else np.nan
                rows.append(row)
                acts.append(np.stack([pooled[l] for l in LAYERS]) if pooled
                            else np.full((len(LAYERS), model.config.hidden_size), np.nan))
            np.save(f"{cdir}/exp{e:03d}_acts.npy", np.stack(acts).astype(np.float16))   # (trials, len(LAYERS), d_model)
            pd.DataFrame(rows).to_csv(csv_path, index=False)   # written last, so resume skips only finished experiments
            if not timed:                                      # time the first experiment this call actually runs
                timed = True
                tqdm.write(f"{name}: {len(rows)} trials per experiment; experiment {e} took {time.time() - t0:.0f} s")

Here, we check that the KV caching for the prompts work. First, we build a block for 200 labeled examples. We check that the activations are similar by computing the cosine similarity with the activations found using the cached prefix and the full prompt.

In [ ]:
# Check: a cached prefix gives the same activations and replies as running each full prompt.
_axes = next(a for a in ALL_CONDITIONS.values() if len(a) == 2)
_idx = np.random.default_rng(1).choice(len(test_sents), N_CONTROL_EXAMPLES + 4, replace=False)
_ex, _replies = [test_sents[i] for i in _idx[:-4]], [test_sents[i] for i in _idx[-4:]]
_Y = np.stack([AXIS_LABELS[la][_idx[:-4]] for la in _axes], axis=1)
_block = [control_messages(_ex, _Y, t, (0, 1)) for t in itertools.product((0, 1), repeat=2)]
_prefix = prefix_cache(_block[0][:-1])

# Activations (replies of different lengths, so the padding is tested too)
_convs = [m + [{"role": "assistant", "content": r + "\n"}] for m, r in zip(_block, _replies)]
_cached, _full = pooled_last_replies(_convs, prefix=_prefix), pooled_last_replies(_convs)
_cos = min(c[l] @ f[l] / (np.linalg.norm(c[l]) * np.linalg.norm(f[l])) for c, f in zip(_cached, _full) for l in LAYERS)
_dz = max(abs(AXIS_FNS[la](c[la[0]][None])[0] - AXIS_FNS[la](f[la[0]][None])[0]) / AXIS_STATS[la][1]
          for c, f in zip(_cached, _full) for la in NEEDED_AXES)
print(f"Cached vs full prompt: min cosine {_cos:.5f}, max axis-score difference {_dz:.3f} SD")
assert _cos > 0.995 and _dz < 0.1, "Cached activations don't match the full prompt; don't run control with the cache."


Run the control task for single axis control and two axes control.

In [ ]:
run_control()

Here, we look at the results for the control task.

In [ ]:
def cohens_d(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    a, b = a[~np.isnan(a)], b[~np.isnan(b)]
    sp = np.sqrt(((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / (len(a) + len(b) - 2))
    return (a.mean() - b.mean()) / sp

def load_control(name):
    files = sorted(glob.glob(f"{control_dir(name)}/exp*.csv"))
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

def matched_cells(d, axes):
    # One row per trial, label order and combination of effective targets: the mean z-score of those replies on
    # each axis. Each row averages ONE unflipped and ONE flipped reply, in single- and two-axis conditions alike,
    # so d means the same thing in every column (and matches the imitation histograms).
    k = len(axes)
    for i, la in enumerate(axes):
        mu, sd = AXIS_STATS[la]
        d[f"z_{i}"] = (d[f"score_{i}"] - mu) / sd
    keys = ["exp", "order"] + [f"eff_{i}" for i in range(k)]
    return d.groupby(keys)[[f"z_{i}" for i in range(k)]].mean().reset_index()

def control_effects(name, mode):
    axes = ALL_CONDITIONS[name]
    k = len(axes)
    d = load_control(name)
    d = d[(d["mode"] == mode) & d["valid"]].copy()
    cell = matched_cells(d, axes)
    out = {"condition": name, "mode": mode, "n_trials": cell["exp"].nunique(),
           "tagged": d["tagged"].mean(), "copied": d["copied"].mean()}
    if k == 1:
        out["d"] = cohens_d(cell[cell["eff_0"] == 1]["z_0"], cell[cell["eff_0"] == 0]["z_0"])
        return out
    for j in range(2):              # score asked 1 vs. 0
        for i in range(2):          # axis measured
            # d within each (other score asked 0/1) x (label order) subset, averaged: each subset has one value per
            # trial and color, built like the single-axis values, so these d's are comparable with the one-axis d.
            ds = [cohens_d(c[c[f"eff_{j}"] == 1][f"z_{i}"], c[c[f"eff_{j}"] == 0][f"z_{i}"])
                  for _, c in cell.groupby([f"eff_{1 - j}", "order"])]
            out[f"target {LABEL_NAMES[j]} → axis {LABEL_NAMES[i]}"] = np.mean(ds)
    together, apart = [], []
    for _, c in cell.groupby("order"):  # within each label order, then averaged
        at = lambda e0, e1: c[(c["eff_0"] == e0) & (c["eff_1"] == e1)]
        together.append(cohens_d(at(1, 1)["z_0"] + at(1, 1)["z_1"], at(0, 0)["z_0"] + at(0, 0)["z_1"]))
        apart.append(cohens_d(at(1, 0)["z_0"] - at(1, 0)["z_1"], at(0, 1)["z_0"] - at(0, 1)["z_1"]))
    out["together: 11 vs 00 on X+Y"] = np.mean(together)
    out["apart: 10 vs 01 on X−Y"] = np.mean(apart)
    return out

control_table = pd.DataFrame([control_effects(n, m) for n in ALL_CONDITIONS for m in CONTROL_MODES])
n_labels = control_table["condition"].map(lambda name: len(ALL_CONDITIONS[name]))
display(control_table[n_labels == 1].dropna(axis=1, how="all").round(2))   # single-label: d
display(control_table[n_labels == 2].dropna(axis=1, how="all").round(2))   # two-label: 2x2 + together/apart

In [ ]:
# Control results as in the paper, built so effect sizes are comparable between single- and two-axis control.
#   plot_pair_hists:  distributions of the imitate-0 and imitate-1 projections, with Cohen's d (as in the paper).
#   plot_pair_deltas: each trial's imitate-1 minus imitate-0 projection (Δ), with the paired t-test.
#   plot_pair_d:      Cohen's d ± 95% CI for every panel of plot_pair_hists, as a bar plot.
from scipy.stats import ttest_1samp
from matplotlib.ticker import MaxNLocator
from matplotlib.patches import Patch

IMIT_COLORS = ("#2a78d6", "#eb6834")      # imitate 0, imitate 1
DELTA_COLOR = "#7b4fc9"                   # imitate 1 − imitate 0
BAR_COLORS = {"single": "#2a78d6", 0: "#eb6834", 1: "#1baf7a"}   # single axis; other axis imitate 0; other axis imitate 1
N_BINS = 40                               # histogram bins across the shared x range

def cell_scores(name, mode):
    # One row per trial, label order and combination of effective targets: the mean raw projection of those
    # replies on each axis (one unflipped + one flipped reply).
    k = len(ALL_CONDITIONS[name])
    d = load_control(name)
    d = d[(d["mode"] == mode) & d["valid"]]
    keys = ["exp", "order"] + [f"eff_{i}" for i in range(k)]
    return d.groupby(keys)[[f"score_{i}" for i in range(k)]].mean().reset_index()

def panels(pair, mode, i):
    # The five panels for axis i of `pair`: single axis first, then two axes, split by what the other axis
    # imitates (0, 1) and whether this axis's label is written first or second in each example.
    axes = ALL_CONDITIONS[pair]
    single_of = {ax[0]: n for n, ax in ALL_CONDITIONS.items() if len(ax) == 1}
    out = [dict(kind="single", cells=cell_scores(single_of[axes[i]], mode) if axes[i] in single_of else None,
                i=0, title="single axis", tick="Single axis")]
    c = cell_scores(pair, mode)
    first = order_name((i, 1 - i))             # the label order with this axis's label written first
    for o in (0, 1):
        for pos in ("first", "second"):
            order = first if pos == "first" else first[::-1]
            out.append(dict(kind=o, pos=pos, cells=c[(c[f"eff_{1 - i}"] == o) & (c["order"] == order)], i=i,
                            title=f"two axes, other axis imitate {o}, this axis {pos}",
                            tick=f"Other axis\nimitate {o},\nthis axis {pos}"))
    return out

def shared_limits(pair, values):
    # Bin edges and the largest bin count over every panel of `pair`, in every control mode.
    # values(cells, i) -> the arrays one panel histograms (one per color). The range always includes 0.
    groups = [v[~np.isnan(v)] for mode in CONTROL_MODES for i in range(2)
              for p in panels(pair, mode, i) if p["cells"] is not None
              for v in values(p["cells"], p["i"])]
    groups = [v for v in groups if len(v)]
    lo, hi = min(0, min(v.min() for v in groups)), max(0, max(v.max() for v in groups))
    pad = 0.03 * (hi - lo)
    bins = np.linspace(lo - pad, hi + pad, N_BINS + 1)
    return bins, max(np.histogram(v, bins)[0].max() for v in groups)

def plot_pair_panels(pair, mode, draw, values, xlabel, fname):
    # 2x5 panels, one row per axis of the pair; columns as in `panels`.
    # draw(ax, cells, i, bins) plots one panel for axis i and returns the text for its title.
    axes = ALL_CONDITIONS[pair]
    bins, top = shared_limits(pair, values)
    fig, grid = plt.subplots(2, 5, figsize=(22, 7.4), dpi=120, sharex=True, sharey=True)
    for i in range(2):
        l, a = axes[i]
        for ax, p in zip(grid[i], panels(pair, mode, i)):
            if p["cells"] is None:
                ax.text(0.5, 0.5, f"No single-axis condition\nfor L{l} {a}", ha="center", va="center",
                        transform=ax.transAxes, color=MUTED)
                ax.tick_params(left=False, labelleft=False)
                ax.spines["left"].set_visible(False)
            else:
                ax.set_title(f"L{l} {a}: {p['title']}\n{draw(ax, p['cells'], p['i'], bins)}",
                             loc="left", color=INK, fontsize=9)
            ax.set_xlabel(xlabel(l, a))
            ax.tick_params(labelbottom=True)                 # x tick labels on both rows
            ax.spines[["top", "right"]].set_visible(False)
        grid[i, 0].set_ylabel("Count")
    grid[0, 0].set_xlim(bins[0], bins[-1])                   # shared by every panel
    grid[0, 0].set_ylim(0, top * 1.22)
    grid[0, 0].yaxis.set_major_locator(MaxNLocator(integer=True))
    handles, names = grid[0, 1].get_legend_handles_labels()
    if handles:
        fig.legend(handles, names, loc="upper left", bbox_to_anchor=(0.01, 1.0), ncol=3, frameon=False, fontsize=9)
    kind = "within-layer" if axes[0][0] == axes[1][0] else "across-layer"
    n = cell_scores(pair, mode)["exp"].nunique()
    fig.suptitle(f"{mode.capitalize()} control, {kind} ({pair}), {n} trials",
                 x=0.01, y=1.06 if handles else 1.03, ha="left", color=INK, fontsize=11)
    fig.tight_layout()
    fig.savefig(f"{SAVE_DIR}/{fname}_{pair}_{mode}.png", bbox_inches="tight")
    plt.show()


# ---- 1. Distributions: imitate 0 vs. imitate 1 ----

def imitation_values(cells, i):
    # Axis i's projections when its own label imitates 0, and 1.
    return [cells[cells[f"eff_{i}"] == e][f"score_{i}"].to_numpy(float) for e in (0, 1)]

def imitation_hist(ax, cells, i, bins):
    # Overlaid histograms of axis i's projection when its own label imitates 0 vs. 1.
    # d: Cohen's d between the two sets of values (how much the histograms overlap), as in the paper.
    v0, v1 = imitation_values(cells, i)
    ax.hist(v0, bins=bins, color=IMIT_COLORS[0], alpha=0.6, label="This axis imitates 0")
    ax.hist(v1, bins=bins, color=IMIT_COLORS[1], alpha=0.6, label="This axis imitates 1")
    ax.axvline(0, color=INK_2, lw=1, ls="--", label="Label boundary (projection = 0)")
    return f"d = {cohens_d(v1, v0):.2f}  ({len(v0)} values per color)"

def plot_pair_hists(pair, mode):
    plot_pair_panels(pair, mode, imitation_hist, imitation_values,
                     lambda l, a: f"L{l} {a} projection", "imitation_hist")


# ---- 2. Paired differences: imitate 1 − imitate 0 within each trial ----

def trial_deltas(cells, i):
    # Axis i's imitate-1 projection minus its imitate-0 projection, one per trial (and label order).
    pv = cells.pivot_table(index=["exp", "order"], columns=f"eff_{i}", values=f"score_{i}").dropna()
    return pv[1] - pv[0]

def delta_values(cells, i):
    return [trial_deltas(cells, i).to_numpy(float)]

def delta_hist(ax, cells, i, bins):
    # Histogram of the Δs, with ▼ at their mean.
    # d_z = mean Δ / SD of Δ (paired effect size). p: one-sample t-test of the trials' Δs against 0.
    delta = trial_deltas(cells, i)
    ax.hist(delta, bins=bins, color=DELTA_COLOR, alpha=0.8)
    ax.axvline(0, color=INK_2, lw=1, ls="--")
    m = delta.mean()
    ax.plot(m, 0.9, marker="v", ms=11, color=INK, zorder=4, transform=ax.get_xaxis_transform())  # x: data, y: axes
    p = ttest_1samp(delta.groupby(level="exp").mean(), 0).pvalue
    return (f"mean Δ (▼) = {m:+.2f}, d_z = {m / delta.std(ddof=1):.2f}\n"
            f"paired p " + ("< 0.001" if p < 0.001 else f"= {p:.3f}") + f", {len(delta)} Δs")

def plot_pair_deltas(pair, mode):
    plot_pair_panels(pair, mode, delta_hist, delta_values,
                     lambda l, a: f"Δ L{l} {a} projection (this axis imitates 1 − imitates 0)",
                     "imitation_delta")


# ---- 3. Cohen's d with 95% CI for every panel ----

def d_ci(d, n1, n2):
    # 95% CI half-width of Cohen's d, as in the paper's code (analysis/stats_fn.py, calc_cohen_d):
    # 1.96 * sqrt((n1 + n2) / (n1 * n2) + d^2 / (2 * (n1 + n2))).
    return 1.96 * np.sqrt((n1 + n2) / (n1 * n2) + d ** 2 / (2 * (n1 + n2)))

def panel_d(cells, i):
    v0, v1 = imitation_values(cells, i)
    d = cohens_d(v1, v0)
    return d, d_ci(d, len(v1), len(v0))

def d_limits():
    # One y range for every bar plot: all bars and their CIs, over every two-axis condition and control mode.
    ext = [d + s * ci for pair, axes in ALL_CONDITIONS.items() if len(axes) == 2
           for mode in CONTROL_MODES for i in range(2)
           for p in panels(pair, mode, i) if p["cells"] is not None
           for d, ci in [panel_d(p["cells"], p["i"])] for s in (-1, 1)]
    lo, hi = min(0, np.nanmin(ext)), max(0, np.nanmax(ext))
    pad = 0.1 * (hi - lo)                                   # room for the value labels
    return lo - pad, hi + pad

def plot_pair_d(pair):
    # Rows: explicit, then implicit control. Columns: the pair's two axes. Bars: the five panels of plot_pair_hists.
    axes = ALL_CONDITIONS[pair]
    modes = [m for m in ("explicit", "implicit") if m in CONTROL_MODES]
    fig, grid = plt.subplots(len(modes), 2, figsize=(12, 4.3 * len(modes)), dpi=120, sharey=True, squeeze=False)
    for row, mode in enumerate(modes):
        for i in range(2):
            ax = grid[row, i]
            ps = panels(pair, mode, i)
            for x, p in enumerate(ps):
                if p["cells"] is None:
                    continue
                d, ci = panel_d(p["cells"], p["i"])
                ax.bar(x, d, 0.7, yerr=ci, color=BAR_COLORS[p["kind"]],
                       hatch="//" if p.get("pos") == "second" else None, edgecolor="white", lw=0,
                       error_kw=dict(ecolor=INK_2, elinewidth=1.2, capsize=3))
                ax.annotate(f"{d:.2f}", (x, d + ci if d >= 0 else d - ci), xytext=(0, 3 if d >= 0 else -3),
                            textcoords="offset points", ha="center", va="bottom" if d >= 0 else "top",
                            fontsize=8, color=INK_2)
            ax.axhline(0, color=BASELINE, lw=0.8)
            ax.set_xticks(range(len(ps)), [p["tick"] for p in ps], fontsize=8)
            ax.set_title(f"{mode.capitalize()} control: L{axes[i][0]} {axes[i][1]}",
                         loc="left", color=INK, fontsize=10)
            ax.grid(axis="y", color=GRID, lw=0.6)
            ax.set_axisbelow(True)
            ax.spines[["top", "right"]].set_visible(False)
        grid[row, 0].set_ylabel("Cohen's d (this axis imitates 1 vs. 0)")
    grid[0, 0].set_ylim(*d_limits())                       # shared by every panel and every bar plot
    # Listed column by column (ncol=3): colors on the first row, this axis's position on the second.
    legend = [Patch(color=BAR_COLORS["single"], label="Single axis"),
              Patch(facecolor="white", edgecolor=INK_2, label="Two axes, this axis first"),
              Patch(color=BAR_COLORS[0], label="Two axes, other axis imitates 0"),
              Patch(facecolor="white", edgecolor=INK_2, hatch="//", label="Two axes, this axis second"),
              Patch(color=BAR_COLORS[1], label="Two axes, other axis imitates 1")]
    fig.legend(handles=legend, loc="upper left", bbox_to_anchor=(0.01, 1.04), ncol=3, frameon=False, fontsize=9)
    kind = "Within-layer" if axes[0][0] == axes[1][0] else "Across-layer"
    fig.suptitle(f"{kind} control ({pair}): Cohen's d", x=0.01, y=1.09, ha="left", color=INK, fontsize=11)
    fig.tight_layout()
    fig.savefig(f"{SAVE_DIR}/control_d_{pair}.png", bbox_inches="tight")
    plt.show()

#### Control task results for WITHIN Layer two-axes control

Figure 3


In [ ]:
plot_pair_d(f"L{WITHIN_LAYER}_PC1+PC2")

#### Control task results for across Layer two-axis control

Figure 4


In [ ]:
plot_pair_d(f"LR_L{ACROSS_LAYERS[0]}+L{ACROSS_LAYERS[1]}")

#### Control task histograms for two-axes control task

Histograms for explicit control (Within layer and across layers)

In [ ]:
plot_pair_hists(f"L{WITHIN_LAYER}_PC1+PC2", "explicit")
plot_pair_hists(f"LR_L{ACROSS_LAYERS[0]}+L{ACROSS_LAYERS[1]}", "explicit")


Histograms for implicit control (Within layer and across layers)

In [ ]:
plot_pair_hists(f"L{WITHIN_LAYER}_PC1+PC2", "implicit")
plot_pair_hists(f"LR_L{ACROSS_LAYERS[0]}+L{ACROSS_LAYERS[1]}", "implicit")